# 06 — Bonus : profils alimentaires, diversité, INCA 2 vs INCA 3, mapping INCA ↔ Ciqual

**Projet NutriHFr** · Questions « Pour aller plus loin »

| Bonus | Méthode |
|---|---|
| Q_Profils | ACP sur la part de l'énergie par groupe d'aliments, puis k-means (k choisi par le score de silhouette) |
| Q_Divers | nombre de groupes consommés par jour vs apport énergétique (corrélation de Spearman, quintiles) |
| Q_Comp_Conso | comparaison agrégée INCA 2 / INCA 3 sur un périmètre harmonisé (18-79 ans) |
| Q_Mapping | appariement approximatif des libellés (difflib), évalué par précision / rappel |

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

In [ ]:
from nutrihfr.pipeline import NutriPipeline
from nutrihfr.stats import describe_weighted, weighted_mean
res = NutriPipeline(verbose=False).run(save=False)
df, lines = res["indiv"], res["lines"]

## 1. Q_Profils — Existe-t-il des profils alimentaires distincts ?

**Variables** : pour chaque individu, la part (%) de son énergie apportée par chaque groupe d'aliments. Travailler en *parts* plutôt qu'en grammes évite que les profils ne reflètent seulement « qui mange beaucoup ».

**Démarche** : standardisation → ACP (pour visualiser et réduire le bruit) → k-means sur les premières composantes → description des groupes.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

e_grp = lines.groupby(["id_indiv", "groupe"])["energie_kcal"].sum().unstack(fill_value=0)
X_pct = 100 * e_grp.div(e_grp.sum(axis=1), axis=0)
X = StandardScaler().fit_transform(X_pct)
pca = PCA().fit(X)
expl = pd.Series(100 * pca.explained_variance_ratio_[:10], index=[f"CP{i+1}" for i in range(10)])
n_comp = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.60) + 1)
print(f"{n_comp} composantes pour 60 % de la variance")
expl.round(1).to_frame("% variance expliquée").T

In [ ]:
Z = PCA(n_components=n_comp, random_state=0).fit_transform(X)
sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=0).fit_predict(Z)) for k in range(2, 7)}
k_best = max(sil, key=sil.get)
print("Silhouette par k :", {k: round(v, 3) for k, v in sil.items()}, "-> k retenu =", k_best)
km = KMeans(k_best, n_init=10, random_state=0).fit(Z)
prof = pd.Series(km.labels_, index=X_pct.index, name="profil")

In [ ]:
# Description : groupes les plus sur-représentés dans chaque profil (écart à la moyenne, en points de %)
ecart = X_pct.groupby(prof).mean() - X_pct.mean()
desc = {p: ", ".join(f"{g} (+{v:.0f} pts)" for g, v in ecart.loc[p].nlargest(3).items()) for p in ecart.index}
taille = df.set_index("id_indiv").join(prof).groupby("profil")["poids_enquete"].sum()
pd.DataFrame({"% population (pondéré)": 100 * taille / taille.sum(), "groupes sur-représentés": pd.Series(desc)})

In [ ]:
pca2 = PCA(n_components=2).fit(X)
P = pca2.transform(X)
fig, ax = plt.subplots(figsize=(8, 6))
for p in sorted(prof.unique()):
    m = prof.to_numpy() == p
    ax.scatter(P[m, 0], P[m, 1], s=8, alpha=0.5, color=PALETTE[p], label=f"profil {p}")
load = pd.DataFrame(pca2.components_.T, index=X_pct.columns, columns=["CP1", "CP2"])
for g in (load ** 2).sum(axis=1).nlargest(8).index:
    ax.annotate("", xy=(4 * load.loc[g, "CP1"], 4 * load.loc[g, "CP2"]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="#52514e", lw=1))
    ax.text(4.3 * load.loc[g, "CP1"], 4.3 * load.loc[g, "CP2"], g, fontsize=8, color="#0b0b0b")
ax.set(title="Biplot ACP : individus colorés par profil, flèches = groupes d'aliments",
       xlabel=f"CP1 ({100 * pca2.explained_variance_ratio_[0]:.0f} %)", ylabel=f"CP2 ({100 * pca2.explained_variance_ratio_[1]:.0f} %)")
ax.legend(); savefig(fig, "06_profils_biplot"); plt.show()

In [ ]:
prof_df = df.set_index("id_indiv").join(prof).reset_index()
prof_df["profil"] = prof_df["profil"].astype(str)
cols = ["energie_kcal", "fibres_g_per1000kcal", "vitc_mg_per1000kcal", "calcium_mg_per1000kcal", "pct_ae_lipides"]
pd.concat({c: describe_weighted(prof_df, c, by="profil").set_index("profil")["moyenne"] for c in cols}, axis=1)

**À retenir** — Un score de silhouette faible (< 0,25) signifie que les profils sont **peu tranchés** : les individus forment un continuum plus que des groupes nets. Il faut le dire honnêtement plutôt que de sur-interpréter les clusters. Nommer chaque profil à partir de ses groupes sur-représentés (ex. « végétal », « sucré-transformé ») et comparer leur qualité nutritionnelle (densités). Limites : k-means suppose des groupes « sphériques » ; le choix de k est en partie subjectif ; une CAH (Ward) peut servir de contrôle.

## 2. Q_Divers — Les gros mangeurs ont-ils une alimentation plus diversifiée ?

In [ ]:
from scipy.stats import spearmanr
div = (lines.groupby(["id_indiv", "id_jour"])["groupe"].nunique().groupby("id_indiv").mean().rename("diversite"))
d = df.set_index("id_indiv").join(div).reset_index()
rho, p = spearmanr(d["energie_kcal"], d["diversite"])
print(f"Spearman énergie ~ diversité : rho = {rho:.2f} (p = {p:.1e})")
d["quintile_energie"] = pd.qcut(d["energie_kcal"], 5, labels=[f"Q{i}" for i in range(1, 6)])
describe_weighted(d, "diversite", by="quintile_energie")[["quintile_energie", "n", "moyenne", "ic95_bas", "ic95_haut"]]

**Attention à l'interprétation** — La relation est en partie **mécanique** : plus on déclare d'aliments, plus l'énergie est élevée, et les sous-déclarants déclarent moins d'aliments. Un indicateur de diversité plus propre compterait les groupes consommés au-dessus d'une quantité minimale, ou utiliserait un score validé (ex. score de diversité alimentaire de la FAO).

## 3. Q_Comp_Conso — Comparer INCA 2 et INCA 3

Comparaison volontairement **limitée aux apports agrégés** des adultes de 18-79 ans, chaque enquête étant pondérée avec **ses propres poids**. Les différences observées mélangent :
- une évolution réelle des comportements (2006 → 2015) ;
- le **changement de méthode** (carnet 7 jours → 3 rappels de 24 h) ;
- les **nomenclatures et tables de composition** différentes ;
- les modes de recrutement et de pondération.

On ne peut donc **pas** attribuer un écart à une évolution réelle (ce que l'Anses souligne elle-même).

In [ ]:
from nutrihfr.loader import Inca2Loader
i2 = Inca2Loader().load()
common = [n for n in C.NUTRIENTS if n in i2.columns and n in df.columns]
both = pd.concat([i2[["enquete", "sexe", "classe_age", "poids_enquete"] + common],
                  df.assign(enquete="INCA3")[["enquete", "sexe", "classe_age", "poids_enquete"] + common]], ignore_index=True)
cmp = pd.concat({n: describe_weighted(both, n, by=["enquete", "sexe"]).set_index(["enquete", "sexe"])[["moyenne", "ic95_bas", "ic95_haut"]]
                 for n in common}, axis=0, names=["nutriment"])
cmp.to_csv(C.TABLES / "comparaison_inca2_inca3.csv")
cmp.unstack("enquete")["moyenne"].round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, n in zip(axes, ["energie_kcal", "fibres_g", "calcium_mg"]):
    t = cmp.loc[n].reset_index()
    for k, s in enumerate(["Homme", "Femme"]):
        ts = t[t.sexe == s].set_index("enquete").loc[["INCA2", "INCA3"]]
        x = np.arange(2) + (k - 0.5) * 0.3
        ax.bar(x, ts["moyenne"], width=0.28, color=SEX_COLORS[s], label=s,
               yerr=[ts["moyenne"] - ts["ic95_bas"], ts["ic95_haut"] - ts["moyenne"]], capsize=3)
    ax.set_xticks(range(2)); ax.set_xticklabels(["INCA 2\n(2006-07)", "INCA 3\n(2014-15)"])
    ax.set(title=C.NUTRIENTS[n][0] + f" ({C.NUTRIENTS[n][1]}/j)")
axes[0].legend(); fig.suptitle("Apports moyens pondérés, adultes 18-79 ans (méthodes de recueil différentes)", y=1.03)
plt.tight_layout(); savefig(fig, "06_inca2_vs_inca3"); plt.show()

## 4. Q_Mapping — Automatiser la correspondance aliments INCA ↔ Ciqual

**Méthode simple et explicable** : normaliser les libellés (minuscules, sans accents ni ponctuation), puis pour chaque aliment INCA choisir le libellé Ciqual le plus proche selon un score de similarité de chaînes (`difflib.SequenceMatcher` + recouvrement de mots). En dessous d'un seuil, l'aliment est laissé **non apparié** (à traiter à la main).

**Évaluation** : on compare à un **jeu de référence** apparié à la main (`data/external/mapping_gold.csv`, ~50 aliments suffisent) :
- *précision* = parmi les appariements proposés, part de corrects ;
- *rappel* = parmi les aliments du jeu de référence, part correctement appariés.

En mode synthétique, la vérité est connue (code Ciqual = code INCA + 20 000).

In [ ]:
import difflib, unicodedata, re
from nutrihfr.loader import CiqualLoader

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9 ]", " ", s).split()

def score(a, b):
    ta, tb = norm(a), norm(b)
    seq = difflib.SequenceMatcher(None, " ".join(ta), " ".join(tb)).ratio()
    jac = len(set(ta) & set(tb)) / max(len(set(ta) | set(tb)), 1)
    return 0.5 * seq + 0.5 * jac

ciq = CiqualLoader().load()
alims = res["tables"]["aliments"]
rows = []
for a in alims.itertuples():
    sc = ciq["libelle"].map(lambda b: score(a.libelle, b))
    j = sc.idxmax()
    rows.append({"code_aliment": a.code_aliment, "libelle_inca": a.libelle, "code_ciqual": ciq.at[j, "code_ciqual"],
                 "libelle_ciqual": ciq.at[j, "libelle"], "score": sc.max()})
mapping = pd.DataFrame(rows)
mapping.sort_values("score").head(8)

In [ ]:
gold_path = ROOT / "data" / "external" / "mapping_gold.csv"
if C.MODE == "synthetic":
    gold = alims[["code_aliment"]].assign(code_ciqual=lambda d: d.code_aliment + 20000)
else:
    gold = pd.read_csv(gold_path)        # colonnes : code_aliment, code_ciqual (validées à la main)

ev = mapping.merge(gold, on="code_aliment", suffixes=("", "_vrai"))
rows = []
for seuil in (0.3, 0.4, 0.5, 0.6, 0.7):
    propose = ev["score"] >= seuil
    correct = propose & (ev["code_ciqual"] == ev["code_ciqual_vrai"])
    rows.append({"seuil": seuil, "proposés": int(propose.sum()), "corrects": int(correct.sum()),
                 "précision": correct.sum() / max(propose.sum(), 1), "rappel": correct.sum() / len(ev)})
pr = pd.DataFrame(rows).set_index("seuil")
pr

In [ ]:
seuil = 0.4
ev["statut"] = np.select([ev.score < seuil, ev.code_ciqual == ev.code_ciqual_vrai], ["non apparié", "correct"], "erroné")
cm = pd.crosstab(ev["statut"], ev["score"] >= seuil, rownames=["résultat"], colnames=["proposé (score ≥ seuil)"])
display(cm)
version = "mapping_v1.0"
mapping.assign(version=version, seuil=seuil, retenu=mapping.score >= seuil).to_csv(
    C.DATA_PROCESSED / f"{version}.csv", index=False)
print(f"Couverture du mapping : {(mapping.score >= seuil).mean():.1%} des aliments INCA -> écrit dans data/processed/{version}.csv")

**À retenir** — Le compromis précision/rappel se règle avec le seuil. Une démarche réaliste est **semi-automatique** : appariement automatique au-dessus d'un seuil élevé (précision ≈ 100 %), revue manuelle en dessous. Pistes : embeddings de phrases (modèle multilingue), utilisation du groupe d'aliments pour restreindre les candidats, codes FoodEx2 communs.

---
Fin des notebooks. Les livrables écrits sont dans `docs/` ; le dashboard se lance avec `streamlit run app/dashboard.py`.